# Grupo 4 — validação sanitária da base climática

Este notebook interpreta `Date Time` no formato dia.mês.ano, verificando nulos, duplicatas, ordenação e intervalos regulares de 10 minutos. A temperatura `T (degC)` é uma variável-alvo possível.

In [1]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'validacao_bases.py').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Não foi possível localizar validacao_bases.py.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from validacao_bases import (
    CONFIGURACOES, calcular_sha256, carregar_base, extrair_datas,
    relatorios_como_dataframe, validar_base,
)

NOME_BASE = 'clima'
config = CONFIGURACOES[NOME_BASE]
dados = carregar_base(NOME_BASE, RAIZ)
print(f'Base: {NOME_BASE} | formato: {dados.shape[0]:,} linhas x {dados.shape[1]} colunas')

Base: clima | formato: 420,551 linhas x 15 colunas


In [2]:
display(dados.head())
display(dados.dtypes.rename('tipo').to_frame())

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


,tipo
Date Time,object
p (mbar),float64
T (degC),float64
Tpot (K),float64
Tdew (degC),float64
rh (%),float64
VPmax (mbar),float64
VPact (mbar),float64
VPdef (mbar),float64
sh (g/kg),float64


## Resultado consolidado

A frequência esperada é de 10 minutos (`10min`). O relatório também indica se as observações precisam ser reordenadas antes da modelagem.

In [3]:
relatorio = validar_base(NOME_BASE, RAIZ)
display(relatorios_como_dataframe({NOME_BASE: relatorio}))
display(pd.Series(relatorio.nulos_por_coluna, name='quantidade_de_nulos').to_frame())

,nome,linhas,colunas,linhas_com_nulos,duplicatas_exatas,datas_invalidas,datas_duplicadas,ordenacao_datas,frequencia_esperada,frequencia_regular,timestamps_ausentes,timestamps_fora_da_grade,aprovada
0,clima,420551,15,0,327,0,327,não ordenada,10min,False,544,0,False


,quantidade_de_nulos


In [4]:
datas = extrair_datas(dados, config)
problemas = dados.loc[dados.duplicated(keep=False) | datas.duplicated(keep=False)].copy()
problemas.insert(0, 'data_normalizada', datas.loc[problemas.index])
print(f'Linhas com duplicidade exata ou temporal: {len(problemas):,}')
display(problemas.head(10))
print('Exemplos de instantes ausentes:', relatorio.exemplos_timestamps_ausentes)
print('Ordenação encontrada:', relatorio.ordenacao_datas)

Linhas com duplicidade exata ou temporal: 654


,data_normalizada,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
78622,2010-07-01 00:10:00,01.07.2010 00:10:00,992.06,17.87,291.69,14.06,78.4,20.50,16.07,4.43,10.14,16.20,1180.21,0.31,0.56,51.11
78623,2010-07-01 00:20:00,01.07.2010 00:20:00,992.02,17.82,291.65,14.03,78.5,20.44,16.04,4.39,10.12,16.17,1180.38,0.23,0.48,52.64
78624,2010-07-01 00:30:00,01.07.2010 00:30:00,992.04,17.92,291.75,14.09,78.3,20.57,16.10,4.46,10.16,16.23,1179.97,0.18,0.40,22.10
78625,2010-07-01 00:40:00,01.07.2010 00:40:00,991.96,17.82,291.65,14.02,78.4,20.44,16.02,4.41,10.11,16.15,1180.32,0.19,0.40,354.80
78626,2010-07-01 00:50:00,01.07.2010 00:50:00,991.90,17.54,291.38,13.96,79.5,20.08,15.96,4.12,10.07,16.10,1181.41,0.24,0.98,21.40
78627,2010-07-01 01:00:00,01.07.2010 01:00:00,991.81,17.35,291.19,13.87,80.0,19.84,15.87,3.97,10.02,16.00,1182.12,0.44,0.96,49.64
78628,2010-07-01 01:10:00,01.07.2010 01:10:00,991.81,17.11,290.95,13.83,81.0,19.54,15.83,3.71,9.99,15.96,1183.11,0.34,0.76,296.10
78629,2010-07-01 01:20:00,01.07.2010 01:20:00,991.85,16.90,290.74,13.74,81.6,19.28,15.74,3.55,9.93,15.87,1184.06,1.21,1.76,239.50
78630,2010-07-01 01:30:00,01.07.2010 01:30:00,991.82,16.87,290.71,13.61,81.1,19.25,15.61,3.64,9.85,15.74,1184.21,1.75,2.28,222.20
78631,2010-07-01 01:40:00,01.07.2010 01:40:00,991.81,16.69,290.53,13.59,81.9,19.03,15.58,3.44,9.83,15.71,1184.94,1.04,1.64,209.90


Exemplos de instantes ausentes: ['2009-10-08T09:50:00', '2009-10-08T10:00:00', '2013-05-16T09:00:00', '2014-07-30T08:10:00', '2014-09-24T17:10:00']
Ordenação encontrada: não ordenada


## Consolidação na granularidade horária

Para a modelagem do grupo 4, os registros são consolidados por hora. Variáveis contínuas usam a média e `max. wv (m/s)` usa o maior valor observado na hora.

In [ ]:
clima_temporal = dados.assign(data=datas).drop_duplicates().set_index('data').sort_index()
colunas_numericas = clima_temporal.select_dtypes(include='number').columns
regras_horarias = {coluna: 'mean' for coluna in colunas_numericas}
regras_horarias['max. wv (m/s)'] = 'max'
clima_horario = clima_temporal.resample(config.frequencia_modelagem).agg(regras_horarias)
print(f'Série horária: {clima_horario.shape[0]:,} linhas x {clima_horario.shape[1]} colunas')
display(clima_horario.head())

## Evidência para o congelamento

O hash identifica exatamente o arquivo analisado. O congelamento completo das cinco bases deve ser feito uma única vez com `congelar_bases('dados_congelados/v1')`.

In [5]:
arquivo = RAIZ / config.caminho
print('Arquivo:', arquivo.relative_to(RAIZ))
print('SHA-256:', calcular_sha256(arquivo))
print('Conclusão:', 'APROVADA' if relatorio.aprovada else 'REQUER TRATAMENTO ANTES DA MODELAGEM')

Arquivo: grupo4\jena_climate_2009_2016.csv
SHA-256: 6fded5f57fe4db37d3b32ac6cdff1e0af769b483cf7ce4cc1ceefd518f45979b
Conclusão: REQUER TRATAMENTO ANTES DA MODELAGEM
